In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [2]:
# Load your dataset (assuming it's in CSV format)
r = 28
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\Testing and Analysis/2024 cleaned data')
match_results = pd.read_csv(f'2024 {r} afl_match_results_cleaned.csv')
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\python scripts\\Catagorical prediction\\k-NN')

# Split features and target
X = match_results.drop(columns=['match.homeTeam.name', 'match.awayTeam.name','venue.name','Margin','Result'])  # Replace 'target_column' with your target column name

# Initialize LabelEncoder
encoder = LabelEncoder()
# Fit and transform the target variable
y = encoder.fit_transform(match_results['Result'])

# Step 1: Split the data into train (80%) and test (20%) sets
train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Ensure no missing or infinite values
assert not X.isnull().values.any(), "Input data contains NaN values."
assert not X.isin([np.inf, -np.inf]).values.any(), "Input data contains infinite values."

# Identify categorical features (you can list their indices or column names)
categorical_features = ['weather.weatherType']  # Replace with your actual categorical feature names
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# OneHotEncode categorical features and scale numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),  # Standardize numerical features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)  # One-hot encode categorical features
    ])

### Run at beginning of season

In [3]:
# import time
# start_time = time.time()

# # Step 2: Time series split for training and hyperparameter tuning on the train set
# tscv = TimeSeriesSplit(n_splits=5)  # Time series split with 5 splits

# # Step 3: Initialize CatBoostClassifier
# knn_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('classifier', KNeighborsClassifier())
# ])

# # Step 4: Hyperparameter tuning using GridSearchCV on the train set
# param_grid = {
#     'classifier__n_neighbors': [3, 5, 7, 9],  # Number of neighbors
#     'classifier__weights': ['uniform', 'distance'],  # Weight function
#     'classifier__p': [1, 2]  # 1: Manhattan, 2: Euclidean distance
# }

# # Use GridSearchCV with time series split on the training data
# grid_search = GridSearchCV(estimator=knn_model, param_grid=param_grid, 
#                            cv=tscv, scoring='accuracy', verbose=1, n_jobs=-1)

# # Step 5: Fit GridSearchCV on the train data
# grid_search.fit(X_train, y_train)

# # Best parameters from GridSearchCV
# best_params = grid_search.best_params_
# print("Best hyperparameters:", best_params)
# print("--- %s seconds ---" % (time.time() - start_time))

### Continue programming

In [4]:
params = {
    'classifier__n_neighbors': 9,  # Number of neighbors
    'classifier__weights': 'uniform',  # Weight function
    'classifier__p': 1  # 1: Manhattan distance
}

# Step 6: Train the model with the best hyperparameters on the training set (using time series splits)
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(params['classifier__n_neighbors'],
                                        weights=params['classifier__weights'],
                                        p=params['classifier__p']))
])

final_model.fit(X_train, y_train)

# Step 7: Test the model on the test set
y_test_pred_probs = final_model.predict_proba(X_test)  # Get the probability for each class on the test set
y_test_pred_class = np.argmax(y_test_pred_probs, axis=1)  # Class with the highest probability

# Evaluate test accuracy
accuracy = accuracy_score(y_test, y_test_pred_class)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.5946


In [5]:
# Step 8: Train on the full dataset (after testing)
final_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['match.homeTeam.Total.kicks',
                                                   'match.homeTeam.Total.handballs',
                                                   'match.homeTeam.Total.disposals',
                                                   'match.homeTeam.Total.marks',
                                                   'match.homeTeam.Total.bounces',
                                                   'match.homeTeam.Total.tackles',
                                                   'match.homeTeam.Total.contestedPossessions',
                                                   'match.homeTeam.Total.uncontest...
                                                   'match.homeTeam.Total.extendedStats.kickEfficiency',
                                                   'match.homeTeam.Total.extendedStats.kickToHandballRatio',
                                                   'match.homeTeam.Total.extendedStats.effectiveDisposals',
                                                   'match.homeTeam.Total.extendedStats.marksOnLead', ...]),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['weather.weatherType'])])),
                ('classifier', KNeighborsClassifier(n_neighbors=9, p=1))])

In [6]:
joblib.dump(final_model, 'knn_model.pkl')
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
with open('accuracy.pkl', 'wb') as f:
    pickle.dump(accuracy, f)
with open('round.pkl', 'wb') as f:
    pickle.dump(r, f)
with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)